# Module 3: Fine-Tuning LLMs

**Goal:** Understand when and how to adapt a pre-trained LLM to your specific task using parameter-efficient methods.

**What you'll learn:**
1. When fine-tuning beats prompt engineering
2. LoRA / QLoRA — how they work and why they're cheap
3. Data preparation for instruction fine-tuning
4. Running a real LoRA fine-tune on a small model
5. Evaluating the fine-tuned model vs. base

> **GPU note:** Full fine-tuning requires GPU. This notebook runs a *tiny* demo locally (CPU-safe), then shows the full QLoRA pipeline you'd run on Colab/Kaggle. Free GPU: [Google Colab](https://colab.research.google.com) · [Kaggle Notebooks](https://www.kaggle.com/code).

---

## 0. Setup

In [ ]:
%pip install -q transformers datasets peft accelerate trl huggingface_hub python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import Dataset
import warnings
warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cpu":
    print("⚠️  Running on CPU — fine-tuning demo will use a tiny model.")
    print("    For real training, run on Colab (Runtime → Change runtime → T4 GPU)")

---
## 1. When to Fine-Tune?

Fine-tuning is expensive. Use it only when other approaches fall short:

| Approach | Cost | When to use |
|----------|------|-------------|
| Prompt engineering | Free | Most tasks — try first |
| Few-shot prompting | Tokens | Consistent format/style |
| RAG | Medium | Domain knowledge, current data |
| **Fine-tuning** | **High** | **Unique style, latency-critical, data privacy** |

**Good use cases for fine-tuning:**
- A model that always responds in your brand's exact tone
- A domain-specific classifier (medical, legal) that must be fast and cheap
- Converting a general model into a structured-output extractor
- Teaching the model a proprietary API / DSL it has never seen

---
## 2. How LoRA Works

Full fine-tuning updates **all** model weights — billions of parameters, massive GPU RAM.

**LoRA** (Low-Rank Adaptation) freezes the original weights and injects small trainable matrices:

```
Original weight matrix W  (frozen, e.g. 4096×4096 = 16M params)
         +
LoRA adapters: W_A (4096×r) × W_B (r×4096)   where r=8 → only 65k params
```

**QLoRA** adds 4-bit quantization of the frozen base model → fits 65B models on a single 24GB GPU.

**Key hyperparameters:**
- `r` (rank): 4–64, higher = more capacity but more params
- `lora_alpha`: scaling factor (typically = r or 2r)
- `target_modules`: which layers to add LoRA to (usually attention: `q_proj`, `v_proj`)

---
## 3. Data Preparation

Instruction fine-tuning needs data in `{instruction, input, output}` format — called the **Alpaca format** (or chat format for multi-turn).

In [ ]:
# Example: teach a model to generate structured bug reports
raw_examples = [
    {
        "instruction": "Convert this bug description into a structured bug report.",
        "input": "The app crashes when I upload a file larger than 10MB on mobile Safari.",
        "output": "**Title:** App crashes on file upload >10MB in mobile Safari\n**Severity:** High\n**Steps to Reproduce:**\n1. Open app on iOS Safari\n2. Navigate to file upload\n3. Select a file >10MB\n**Expected:** File uploads successfully\n**Actual:** App crashes\n**Environment:** iOS Safari, file >10MB"
    },
    {
        "instruction": "Convert this bug description into a structured bug report.",
        "input": "Login button doesn't work on first click, need to click twice.",
        "output": "**Title:** Login button requires double-click\n**Severity:** Medium\n**Steps to Reproduce:**\n1. Navigate to login page\n2. Enter credentials\n3. Click login button once\n**Expected:** User logged in after single click\n**Actual:** First click ignored, second click works\n**Environment:** All browsers"
    },
    {
        "instruction": "Convert this bug description into a structured bug report.",
        "input": "Dark mode makes text invisible on the settings page.",
        "output": "**Title:** Text invisible in dark mode on Settings page\n**Severity:** High\n**Steps to Reproduce:**\n1. Enable dark mode\n2. Navigate to Settings\n**Expected:** All text readable\n**Actual:** Text color matches background — invisible\n**Environment:** Dark mode enabled"
    },
]

def format_alpaca(example: dict) -> str:
    """Format to Alpaca instruction template."""
    if example["input"]:
        return (
            f"### Instruction:\n{example['instruction']}\n\n"
            f"### Input:\n{example['input']}\n\n"
            f"### Response:\n{example['output']}"
        )
    return (
        f"### Instruction:\n{example['instruction']}\n\n"
        f"### Response:\n{example['output']}"
    )

formatted = [format_alpaca(ex) for ex in raw_examples]
print("Example formatted training sample:")
print("─" * 60)
print(formatted[0])

In [ ]:
# Convert to HuggingFace Dataset
dataset = Dataset.from_dict({"text": formatted})
print(f"Dataset size: {len(dataset)} examples")
print(f"\nDataset features: {dataset.features}")

# In production you'd split train/eval
splits = dataset.train_test_split(test_size=0.1, seed=42)
print(f"Train: {len(splits['train'])}, Eval: {len(splits['test'])}")

---
## 4. LoRA Configuration

The cell below shows the real configuration you'd use for fine-tuning a 7B model on Colab.

In [ ]:
from peft import LoraConfig, TaskType
from transformers import TrainingArguments

# LoRA adapter config
lora_config = LoraConfig(
    r=16,                          # Rank: higher = more capacity, more params
    lora_alpha=32,                 # Scaling (usually 2x rank)
    target_modules=[               # Which weight matrices to add adapters to
        "q_proj", "k_proj",
        "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Training arguments
training_args = TrainingArguments(
    output_dir="./fine_tuned_model",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # Effective batch = 4×4 = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    save_strategy="epoch",
    logging_steps=10,
    fp16=True,                      # Mixed precision (GPU only)
    report_to="none",
)

print("LoRA config:")
print(f"  rank (r): {lora_config.r}")
print(f"  alpha: {lora_config.lora_alpha}")
print(f"  target modules: {lora_config.target_modules}")
print(f"\nNote: For a 7B model, LoRA adds ~0.1% extra trainable params")
print(f"  Full FT: ~7,000M params | LoRA: ~7M params")

---
## 5. Full Training Pipeline (GPU required)

The block below shows the complete QLoRA training loop. It's commented out to avoid errors on CPU. Run it on Colab with a T4/A100.

In [ ]:
# ─── FULL QLORA TRAINING (requires GPU + bitsandbytes) ────────────────────────
# Uncomment and run on Google Colab (Runtime → Change runtime type → T4 GPU)

"""
from transformers import BitsAndBytesConfig
from peft import get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

MODEL_ID = "mistralai/Mistral-7B-v0.1"   # or "meta-llama/Llama-3.2-3B"

# 4-bit quantisation config (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load base model in 4-bit
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

# Prepare for k-bit training, apply LoRA
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()   # should be ~0.1-0.5%

# Train
trainer = SFTTrainer(
    model=model,
    train_dataset=splits["train"],
    eval_dataset=splits["test"],
    dataset_text_field="text",
    max_seq_length=512,
    args=training_args,
    peft_config=lora_config,
)
trainer.train()
trainer.save_model("./fine_tuned_model")
"""
print("Full training pipeline ready — uncomment to run on GPU.")

---
## 6. Local Demo: Tokenization & Forward Pass

Even without a GPU we can verify data flows through a tiny model correctly.

In [ ]:
# Use a tiny model (gpt2) to verify the pipeline works on CPU
MODEL_ID = "gpt2"   # 124M params — downloads in seconds

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

sample = formatted[0]
tokens = tokenizer(sample, return_tensors="pt", truncation=True, max_length=256)

print(f"Input text length: {len(sample)} chars")
print(f"Token count: {tokens['input_ids'].shape[1]}")
print(f"Token IDs (first 10): {tokens['input_ids'][0][:10].tolist()}")
print(f"Decoded back: '{tokenizer.decode(tokens['input_ids'][0][:15])}...'")

In [ ]:
from peft import get_peft_model

# Apply LoRA to GPT-2 (CPU-safe demo)
small_lora = LoraConfig(
    r=4,
    lora_alpha=8,
    target_modules=["c_attn"],   # GPT-2 attention layer name
    task_type=TaskType.CAUSAL_LM,
)

base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
lora_model = get_peft_model(base_model, small_lora)
lora_model.print_trainable_parameters()

---
## 7. After Training: Using Your Fine-Tuned Model

In [ ]:
# Loading a saved LoRA adapter (after GPU training)
"""
from peft import PeftModel

# Load base model + merge adapters
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto")
model = PeftModel.from_pretrained(base, "./fine_tuned_model")

# Optional: merge weights into base for faster inference
model = model.merge_and_unload()

# Push to HuggingFace Hub
model.push_to_hub("your-username/my-finetuned-model")
tokenizer.push_to_hub("your-username/my-finetuned-model")
"""

print("After training, push to HuggingFace Hub with model.push_to_hub()")

---
## 8. Fine-Tuning Decision Checklist

Before spending GPU hours, answer these:

- [ ] Did I try prompt engineering first and it's not good enough?
- [ ] Do I have ≥500 high-quality examples (ideally 1k–10k)?
- [ ] Is the task consistent enough to be learnable from examples?
- [ ] Will the model be called frequently enough to justify the compute cost?
- [ ] Have I established a baseline eval metric to measure improvement?

If you checked all 5 — fine-tune. Otherwise, keep iterating on prompts.

## 🧪 Exercises

1. **Data quality experiment**: Add 2 deliberately bad training examples (wrong format). Does it hurt output quality? By how much?
2. **Rank ablation**: Try `r=4` vs `r=64` on Colab. Compare loss curves and output quality.
3. **Dataset format**: Convert the examples above to ShareGPT (multi-turn chat) format instead of Alpaca. Which format does your use case need?
4. **Existing dataset**: Load `tatsu-lab/alpaca` from HuggingFace — how is it formatted? Would it be useful for your task?

---
**Next:** [Module 04 — Evaluation](../04-evaluation/evaluation.ipynb)